In [50]:
# --- SECTION 1: LOAD DROUGHT FORECAST DATA ---

import pandas as pd

# Updated path for the drought forecast dataset
# Switched from local storage to GitHub raw URL for team accessibility
file_path = 'https://github.com/alhelibrito/Terra_Project/raw/main/input_files/USGS_streamflow_drought_forecasts_2026-04-12.parquet'

df_forecasts = pd.read_parquet(file_path)

# Verify the data loaded correctly
print(f"Dataset loaded from: {file_path}")
print(df_forecasts.head())

Dataset loaded from: https://github.com/alhelibrito/Terra_Project/raw/main/input_files/USGS_streamflow_drought_forecasts_2026-04-12.parquet
  source model_type  issue_date     StaID forecast_date  forecast_week  \
0   USGS    LSTM<50  2026-04-12  01019000    2026-04-19              1   
1   USGS    LSTM<50  2026-04-12  01019000    2026-04-26              2   
2   USGS    LSTM<50  2026-04-12  01019000    2026-05-03              3   
3   USGS    LSTM<50  2026-04-12  01019000    2026-05-10              4   
4   USGS    LSTM<50  2026-04-12  01019000    2026-05-17              5   

   median_pct  pred_interv_05_pct  pred_interv_95_pct  
0   12.240296            3.042142           27.355007  
1   17.813828            4.493113           38.486790  
2   13.843739            1.638931           42.612217  
3   19.402218            3.441107           43.027023  
4   19.324324            3.392919           44.082897  


In [51]:
# ==============================================================================
# SECTION 2: METADATA RETRIEVAL & COORDINATE MAPPING
# Description: Fetch geographic metadata (lat/long) from the USGS NWIS API.
#              Processing is done in batches of 500 to comply with URL limits.
# ==============================================================================

# Create an empty list to store metadata chunks
all_chunks = []

# Process stations in batches of 500 to avoid API URL length limits
batch_size = 500

for i in range(0, len(sta_ids_list), batch_size):
    batch = sta_ids_list[i:i + batch_size]
    print(f"Fetching batch: {i} to {i + len(batch)}...")
    
    # Request metadata for the current batch using dataretrieval.nwis
    # df_chunk contains site info, '_' ignores the second return value (metadata info)
    df_chunk, _ = nwis.get_info(sites=batch)
    all_chunks.append(df_chunk)

# Combine all batches into a single DataFrame for mapping
df_metadata = pd.concat(all_chunks, ignore_index=True)

# --- DATA RESTRUCTURING ---
# Filter relevant columns and rename for consistency with the main dataset
df_locations = df_metadata[['site_no', 'station_nm', 'dec_lat_va', 'dec_long_va']].copy()
df_locations.columns = ['StaID', 'station_name', 'latitude', 'longitude']

# Verification of the retrieval process
print("-" * 30)
print(f"Success! Retrieved coordinates for {len(df_locations)} stations.")
print(df_locations.head())

Fetching batch: 0 to 500...
Fetching batch: 500 to 1000...
Fetching batch: 1000 to 1500...
Fetching batch: 1500 to 2000...
Fetching batch: 2000 to 2500...
Fetching batch: 2500 to 2939...
------------------------------
Success! Retrieved coordinates for 2939 stations.
      StaID                                   station_name   latitude  \
0  01019000  Grand Lake Stream at Grand Lake Stream, Maine  45.172500   
1  01022500        Narraguagus River at Cherryfield, Maine  44.608056   
2  01031500   Piscataquis River near Dover-Foxcroft, Maine  45.175000   
3  01034500         Penobscot River at West Enfield, Maine  45.236111   
4  01042500             Kennebec River at The Forks, Maine  45.339722   

   longitude  
0 -67.768889  
1 -67.935278  
2 -69.314722  
3 -68.651389  
4 -69.961944  


In [52]:
# ==============================================================================
# SECTION 3: DATA EXPORT - STATIONS INVENTORY
# Description: Generate a standardized CSV file containing all validated 
#              USGS stations with their respective geographic coordinates.
# ==============================================================================

# 1. Define the output path for the processed inventory
# This file will serve as the reference for future spatial joins with Datacenters
output_csv_path = r'D:\Alheli\2026\Terra.do\Data\RDC_Stations_Inventory.csv'

# 2. Export the DataFrame to CSV
# 'index=False' ensures the file is clean and ready for database ingestion
df_locations.to_csv(output_csv_path, index=False)

# --- FINAL EXPORT VERIFICATION ---
print(f"Success! Master CSV generated at: {output_csv_path}")
print(f"Total stations exported: {len(df_locations)}")
print("-" * 40)
print("Preview of Final Exported Schema:")
print(df_locations[['StaID', 'station_name', 'latitude', 'longitude']].head())

Success! Master CSV generated at: D:\Alheli\2026\Terra.do\Data\RDC_Stations_Inventory.csv
Total stations exported: 2939
----------------------------------------
Preview of Final Exported Schema:
      StaID                                   station_name   latitude  \
0  01019000  Grand Lake Stream at Grand Lake Stream, Maine  45.172500   
1  01022500        Narraguagus River at Cherryfield, Maine  44.608056   
2  01031500   Piscataquis River near Dover-Foxcroft, Maine  45.175000   
3  01034500         Penobscot River at West Enfield, Maine  45.236111   
4  01042500             Kennebec River at The Forks, Maine  45.339722   

   longitude  
0 -67.768889  
1 -67.935278  
2 -69.314722  
3 -68.651389  
4 -69.961944  


In [53]:
# ==============================================================================
# SECTION 4: DATACENTER INVENTORY INGESTION
# Description: Load datacenter spatial data from GitHub repository for 
#              proximity mapping against USGS drought stations.
# ==============================================================================

import pandas as pd

# Direct link to the raw CSV file on GitHub
dc_url = 'https://raw.githubusercontent.com/alhelibrito/Terra_Project/main/input_files/datacenters.csv'

# Load dataset and select critical columns for spatial analysis
try:
    df_raw_dc = pd.read_csv(dc_url)
    
    # Keeping only spatial and capacity identifiers
    cols = ['Name', 'Operator', 'State', 'City', 'Power', 'Latitude', 'Longitude']
    df_dc_clean = df_raw_dc[cols].copy()
    
    print(f" Success: Loaded {len(df_dc_clean)} datacenters.")
    print(df_dc_clean.head())

except Exception as e:
    print(f"x Connection Error: {e}")

 Success: Loaded 4682 datacenters.
                         Name          Operator  State           City  \
0          Helios Data Center            Galaxy  Texas        Dickens   
1  Fort Stockton Clean Campus           Lancium  Texas  Fort Stockton   
2             Project Horizon       Poolside AI  Texas  Fort Stockton   
3     Data Journey The Bunker      Data Journey  Texas  Montgomery Tx   
4       MDC Eagle Pass - EGP1  MDC Data Centers  Texas     Eagle Pass   

     Power   Latitude   Longitude  
0   800 MW  33.780636 -100.867100  
1   325 MW  30.905623 -102.917046  
2  2000 MW  30.963485 -102.848175  
3   200 MW  30.376981  -95.669244  
4   0.5 MW  28.702367 -100.502761  


In [54]:
# ==============================================================================
# SECTION 5: PROXIMITY ANALYSIS
# Description: Identifying the nearest USGS drought station for each Datacenter
#              using the Haversine formula (Earth's curvature distance).
# ==============================================================================

from sklearn.neighbors import BallTree
import numpy as np

# 1. Convert coordinates to Radians for the BallTree algorithm
stations_rad = np.deg2rad(df_locations[['latitude', 'longitude']].values)
dc_rad = np.deg2rad(df_dc_clean[['Latitude', 'Longitude']].values)

# 2. Build the spatial index with the stations
# metric='haversine' ensures calculation on a sphere
tree = BallTree(stations_rad, metric='haversine')

# 3. Query the nearest station for each Datacenter (k=1)
# distances are in radians, we multiply by Earth's radius (6371 km)
dist_rad, indices = tree.query(dc_rad, k=1)
earth_radius_km = 6371

# 4. Create the final mapping dataframe
df_final_mapping = df_dc_clean.copy()
df_final_mapping['nearest_station_id'] = df_locations['StaID'].iloc[indices.flatten()].values
df_final_mapping['distance_km'] = dist_rad.flatten() * earth_radius_km

print(" Spatial Mapping Complete!")
print(f"Sample of Datacenters and their closest Monitoring Station:")
print(df_final_mapping[['Name', 'nearest_station_id', 'distance_km']].head())

 Spatial Mapping Complete!
Sample of Datacenters and their closest Monitoring Station:
                         Name nearest_station_id  distance_km
0          Helios Data Center           08082000    76.570763
1  Fort Stockton Clean Campus           08405500   187.992813
2             Project Horizon           08405500   187.213000
3     Data Journey The Bunker           08068000    25.119162
4       MDC Eagle Pass - EGP1           08192000    75.486194


In [57]:
# ==============================================================================
# SECTION 6: DROUGHT RISK INTEGRATION & TIME-SERIES EXPORT
# Description: Merges spatial infrastructure mapping with RDC model forecasts 
#              to establish a 13-week risk timeline. Categorization is aligned 
#              with USGS streamflow drought thresholds and policy triggers.
# ==============================================================================

# 1. Spatial Join: Mapping Infrastructure to Forecast Sites
# Infrastructure locations are merged with the nearest USGS station forecasts.
df_risk_assessment = pd.merge(
    df_final_mapping, 
    df_forecasts, 
    left_on='nearest_station_id', 
    right_on='StaID', 
    how='inner'
)

# 2. Time-Series Data Preparation & Cleaning
# Sorting ensures a sequential 13-week outlook. Duplicates are removed to 
# ensure a single, clean record per forecast date per site.
df_final_export = df_risk_assessment.sort_values(by=['Name', 'forecast_date']).copy()
df_final_export = df_final_export.drop_duplicates(subset=['Name', 'forecast_date'], keep='first')

# 3. Drought Risk Categorization Logic (USGS & Policy Aligned)
# Thresholds translate statistical percentiles into actionable risk levels:
# < 10%: Extreme
# < 20%: High Risk 
# < 40%: Warning 
def get_risk_status(pct):
    if pct < 10: return "Extreme Risk"
    elif pct < 20: return "High Risk"
    elif pct < 40: return "Warning"
    else: return "Safe"

# Identification of the median percentile column for risk labeling
median_col = 'median_pct' if 'median_pct' in df_final_export.columns else 'p50'
df_final_export['Risk_Status'] = df_final_export[median_col].apply(get_risk_status)

# 4. Global Analysis Statistics (Immediate Horizon)
# Calculation of current critical alerts based on the first available forecast week.
df_current_week = df_final_export.drop_duplicates(subset=['Name'], keep='first')
critical_count = len(df_current_week[df_current_week[median_col] < 20])

print("Integration process complete.")
print(f"Total unique infrastructure sites: {len(df_current_week)}")
print(f"Total cleaned time-series records: {len(df_final_export)}")
print(f"Immediate critical/high risk alerts: {critical_count}")
print("-" * 40)

# 5. Multidimensional Probability Export
# Dynamic column selection to include metadata and all available percentiles.
metadata_cols = [
    'Name', 'Operator', 'State', 'City', 'Power', 'Latitude', 'Longitude',
    'nearest_station_id', 'distance_km', 'forecast_date', 'Risk_Status'
]

# Detection of percentile columns (p5 through p95) for uncertainty analysis
prob_cols = [col for col in df_final_export.columns if (col.startswith('p') and any(char.isdigit() for char in col)) or col == 'median_pct']
final_columns = [col for col in metadata_cols if col in df_final_export.columns] + sorted(list(set(prob_cols)))

# Final CSV Export 
df_final_export[final_columns].to_csv('DataCenter_USGS_Drought_Forecast.csv', index=False)
print("File 'Infrastructure_Drought_Forecast_Timeline.csv' generated successfully.")

# Preview of the finalized 13-week trajectory
print(df_final_export[final_columns].head(13))

Integration process complete.
Total unique infrastructure sites: 4676
Total cleaned time-series records: 60788
Immediate critical/high risk alerts: 2118
----------------------------------------
File 'Infrastructure_Drought_Forecast_Timeline.csv' generated successfully.
             Name                      Operator     State      City  Power  \
117494  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117495  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117496  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117497  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117498  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117499  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117500  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117501  1 Ramland  1547 Critical Systems Realty  New York  New York  12 MW   
117502  1 Ramland  1547 Crit

In [58]:
pd.set_option('display.max_columns', None)

# Preview of the finalized data
df_final_for_map.head()

,Name,Operator,State,City,Power,Latitude,Longitude,nearest_station_id,distance_km,source,model_type,issue_date,StaID,forecast_date,forecast_week,median_pct,pred_interv_05_pct,pred_interv_95_pct,Risk_Status
0,Helios Data Center,Galaxy,Texas,Dickens,800 MW,33.780636,-100.867100,08082000,76.570763,USGS,LSTM<50,2026-04-12,08082000,2026-04-19,1,10.352182,2.835176,37.200684,High Risk
26,Fort Stockton Clean Campus,Lancium,Texas,Fort Stockton,325 MW,30.905623,-102.917046,08405500,187.992813,USGS,LSTM<50,2026-04-12,08405500,2026-04-19,1,1.716381,0.463654,12.915097,High Risk
52,Project Horizon,Poolside AI,Texas,Fort Stockton,2000 MW,30.963485,-102.848175,08405500,187.213000,USGS,LSTM<50,2026-04-12,08405500,2026-04-19,1,1.716381,0.463654,12.915097,High Risk
78,Data Journey The Bunker,Data Journey,Texas,Montgomery Tx,200 MW,30.376981,-95.669244,08068000,25.119162,USGS,LSTM<50,2026-04-12,08068000,2026-04-19,1,30.533310,12.702352,46.517754,Warning
104,MDC Eagle Pass - EGP1,MDC Data Centers,Texas,Eagle Pass,0.5 MW,28.702367,-100.502761,08192000,75.486194,USGS,LSTM<50,2026-04-12,08192000,2026-04-19,1,9.006710,4.081707,24.161007,High Risk
